# The Math Behind Augmented Reality Overlays

**CS474: Human Computer Interaction — Augmented Reality**

Augmented reality apps draw virtual content so it appears "attached" to a real surface — a poster, a table, a marker card.  The core trick is a **homography**: a 3x3 matrix that maps points from a flat overlay image onto the (perspective-distorted) quadrilateral where that surface appears in the camera frame.

In this notebook you will:

1. Build a simulated "camera view" containing a tilted marker (no camera required)
2. Compute a homography from 4 point correspondences — with plain `numpy`, so you can see exactly what libraries like OpenCV do for you
3. Warp an overlay image onto the marker so it appears glued to the scene

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(474)

## Part 1: The Scene and the Marker

Our "camera frame" is a 480x360 image.  Somewhere in it, a flat rectangular marker is seen in perspective — so its corners form an arbitrary quadrilateral rather than a rectangle.  In a real AR pipeline, a detector (e.g., ArUco markers in OpenCV) finds these 4 corner points for you.

In [ ]:
FRAME_W, FRAME_H = 480, 360

# A simple background scene: soft gradient + noise, as a stand-in for a camera image
yy, xx = np.mgrid[0:FRAME_H, 0:FRAME_W]
scene = 0.4 + 0.3 * (yy / FRAME_H) + rng.normal(0, 0.02, (FRAME_H, FRAME_W))
scene = np.dstack([scene * 0.9, scene, scene * 1.1]).clip(0, 1)  # bluish tint

# Where the detector "found" the marker corners in the frame,
# in order: top-left, top-right, bottom-right, bottom-left
marker_corners = np.array([
    [150,  90],
    [330, 120],
    [310, 260],
    [120, 230],
], dtype=float)

plt.figure(figsize=(6, 4.5))
plt.imshow(scene)
poly = plt.Polygon(marker_corners, fill=False, edgecolor='red', lw=2)
plt.gca().add_patch(poly)
plt.title('Camera frame with detected marker (red)')
plt.show()

## Part 2: The Overlay Image

Let's make a small overlay to project onto the marker — a simple "info card" drawn with numpy.

In [ ]:
OV_W, OV_H = 200, 150
overlay = np.ones((OV_H, OV_W, 3))
overlay[:, :] = [0.13, 0.35, 0.60]            # card background
overlay[10:40, 10:190] = [0.95, 0.80, 0.20]   # title bar
for row in range(60, 140, 20):                # "text" lines
    overlay[row:row+8, 15:185] = [0.92, 0.92, 0.92]

plt.figure(figsize=(3, 2.2))
plt.imshow(overlay); plt.title('Overlay (in its own coordinates)'); plt.axis('off')
plt.show()

## Part 3: Compute the Homography

A homography H maps overlay coordinates \((x, y)\) to frame coordinates \((x', y')\):

$$ s \begin{bmatrix} x' \\ y' \\ 1 \end{bmatrix} = H \begin{bmatrix} x \\ y \\ 1 \end{bmatrix} $$

Each of the 4 corner correspondences gives 2 linear equations in the 9 entries of H; we solve the resulting system with the SVD (this is the standard *Direct Linear Transform*).  This is exactly what `cv2.findHomography` computes.

In [ ]:
def find_homography(src, dst):
    """DLT: solve for H such that dst ~ H @ src (4+ point pairs)."""
    A = []
    for (x, y), (xp, yp) in zip(src, dst):
        A.append([-x, -y, -1,  0,  0,  0, x*xp, y*xp, xp])
        A.append([ 0,  0,  0, -x, -y, -1, x*yp, y*yp, yp])
    A = np.array(A)
    _, _, Vt = np.linalg.svd(A)
    H = Vt[-1].reshape(3, 3)
    return H / H[2, 2]

overlay_corners = np.array([[0, 0], [OV_W - 1, 0], [OV_W - 1, OV_H - 1], [0, OV_H - 1]], float)
H = find_homography(overlay_corners, marker_corners)
np.set_printoptions(precision=3, suppress=True)
print(H)

## Part 4: Warp the Overlay into the Scene

To avoid holes, we work *backwards*: for every pixel inside the marker quadrilateral, we apply the **inverse** homography to ask "which overlay pixel lands here?"  (This inverse mapping is how `cv2.warpPerspective` works internally.)

In [ ]:
Hinv = np.linalg.inv(H)

composite = scene.copy()
ys, xs = np.mgrid[0:FRAME_H, 0:FRAME_W]
pts = np.stack([xs.ravel(), ys.ravel(), np.ones(xs.size)])   # homogeneous frame coords
src = Hinv @ pts
src /= src[2]                                                # divide by scale s
sx, sy = src[0].reshape(FRAME_H, FRAME_W), src[1].reshape(FRAME_H, FRAME_W)

inside = (sx >= 0) & (sx < OV_W - 1) & (sy >= 0) & (sy < OV_H - 1)
composite[inside] = overlay[sy[inside].astype(int), sx[inside].astype(int)]

plt.figure(figsize=(6, 4.5))
plt.imshow(composite)
plt.title('Overlay warped onto the marker: basic AR!')
plt.show()

## Your Turn

1. **Move the marker.**  Change `marker_corners` to simulate the camera viewing the marker from a steeper angle.  When does the overlay become hard to read?  What does that imply for where AR apps should *anchor* text?
2. **Blend, don't replace.**  Modify Part 4 to alpha-blend the overlay at 70% opacity instead of overwriting the scene.  When is a translucent overlay better UX than an opaque one (think: field of vision, safety)?
3. **Design connection.**  The activity goal is "dynamic presentation of *unobtrusive* information in one's field of vision."  List three rules you would adopt for what AR content may cover, based on what you observed here.

## Reflection

In the AR programming assignment you'll use OpenCV, which detects the marker and computes this same homography in real time — now you know what those library calls are actually doing.